<a href="https://colab.research.google.com/github/abhijadhav14/Data-Analytics-Using-Python/blob/main/Random_Forest_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Random Forest Regression using PySpark MLlib

# Import Libraries
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize Spark Session
spark = SparkSession.builder.appName("RandomForestExample").getOrCreate()

# Load Dataset from Google Drive
gdrive_csv_path = '/content/drive/MyDrive/housing_large.csv'
df = spark.read.csv(gdrive_csv_path, header=True, inferSchema=True)

# Combine Features into a Single Vector
assembler = VectorAssembler(inputCols=["sqft", "bedrooms"], outputCol="features")
df_assembled = assembler.transform(df)

# Select Features and Target
final_df = df_assembled.select("features", "price")

# Split Data
train_data, test_data = final_df.randomSplit([0.8, 0.2], seed=42)

# Create and Train Random Forest Regression Model
# numTrees specifies the number of trees in the forest
rf = RandomForestRegressor(featuresCol="features", labelCol="price", numTrees=10, seed=42)
rf_model = rf.fit(train_data)

# Make Predictions
predictions = rf_model.transform(test_data)

# Evaluate Model (Using R-Squared)
evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="r2")
r2 = evaluator.evaluate(predictions)
print("R-Squared (R2):", r2)

# View Example Predictions
print("\nPredictions for Test Data:")
predictions.select("features", "price", "prediction").show(truncate=False)

R-Squared (R2): 0.8976629077972579

Predictions for Test Data:
+------------+------------------+------------------+
|features    |price             |prediction        |
+------------+------------------+------------------+
|[1000.0,2.0]|151524.69161513733|273188.60022502614|
|[1000.0,2.0]|256778.33087646787|273188.60022502614|
|[1000.0,3.0]|191196.08818737933|279552.60729630856|
|[1000.0,3.0]|285914.91330812505|279552.60729630856|
|[1000.0,4.0]|199791.6665693316 |278274.9361525852 |
|[1000.0,4.0]|308087.26944694424|278274.9361525852 |
|[1000.0,5.0]|321884.90206810547|288503.61694539   |
|[1001.0,2.0]|251538.30953848225|273188.60022502614|
|[1001.0,5.0]|195890.67779215134|288503.61694539   |
|[1001.0,5.0]|230360.02964329987|288503.61694539   |
|[1001.0,5.0]|262876.98745818774|288503.61694539   |
|[1001.0,5.0]|270931.996387155  |288503.61694539   |
|[1001.0,5.0]|397638.1653484768 |288503.61694539   |
|[1002.0,2.0]|191163.49543836567|273188.60022502614|
|[1002.0,3.0]|132582.4429926811 |279

First, let's generate a large synthetic dataset and save it to a CSV file. This dataset will mimic the structure of the dummy data used in the existing PySpark example.

In [1]:
import numpy as np
import pandas as pd

# Generate a large synthetic dataset
num_samples = 100000 # Increased number of samples for a 'large' dataset

# Generate 'sqft' values between 1000 and 5000
sqft = np.random.randint(1000, 5000, num_samples)

# Generate 'bedrooms' values between 2 and 6
bedrooms = np.random.randint(2, 6, num_samples)

# Generate 'price' based on sqft and bedrooms with some noise
# Simple linear relationship: price = sqft * 150 + bedrooms * 20000 + noise
noise = np.random.normal(0, 50000, num_samples)
price = sqft * 150 + bedrooms * 20000 + noise

# Ensure prices are positive
price[price < 0] = 100000 # Set a minimum price if it goes below zero

data_large = pd.DataFrame({
    'sqft': sqft,
    'bedrooms': bedrooms,
    'price': price
})

# Display the first few rows of the generated data
print("Generated large dataset head:")
display(data_large.head())

# Save the dataset to a CSV file in a temporary location
# We'll later move this to Google Drive or read it from there after mounting
csv_file_path = '/tmp/housing_large.csv'
data_large.to_csv(csv_file_path, index=False)
print(f"\nLarge dataset saved to {csv_file_path}")

Generated large dataset head:


,sqft,bedrooms,price
0,2550,5,485380.131205
1,1536,3,321618.814942
2,3987,5,640295.874560
3,1541,5,215215.802592
4,3774,5,557000.945434



Large dataset saved to /tmp/housing_large.csv


In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Define the path where you want to save/read the CSV in Google Drive
gdrive_csv_path = '/content/drive/MyDrive/housing_large.csv'

# Copy the generated CSV from temporary storage to Google Drive
import shutil
shutil.copy(csv_file_path, gdrive_csv_path)
print(f"CSV file copied to Google Drive: {gdrive_csv_path}")

Mounted at /content/drive
CSV file copied to Google Drive: /content/drive/MyDrive/housing_large.csv


Finally, I will modify the existing PySpark cell to load this large dataset from Google Drive. Please run the modified cell to use the new dataset.